# GNN Retrieval Training from Neo4j (`Chunk`-`Entity`)

Notebook строит обучающий пайплайн для retrieval на графе Neo4j:
- читает структуру `(:Chunk)-[:MENTIONS]->(:Entity)` и `(:Entity)-[:RELATED_TO]->(:Entity)`,
- формирует гетерогенный граф,
- обучает GraphSAGE для ранжирования `Entity` по `Chunk` (link prediction),
- считает `Recall@K`/`MRR`,
- записывает `gnn_embedding` обратно в Neo4j.


In [1]:
# При необходимости раскомментируйте установку зависимостей
%pip install neo4j pandas numpy scikit-learn torch torch-geometric qdrant-client



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [26]:
import os
import random

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from neo4j import GraphDatabase
from qdrant_client import QdrantClient
from qdrant_client.models import FieldCondition, Filter, MatchAny
from torch import nn
from torch_geometric.data import HeteroData
from torch_geometric.nn import HeteroConv, SAGEConv

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
DEVICE


device(type='cuda')

In [27]:
# Конфиг подключения
NEO4J_URI = os.getenv('NEO4J_URI', 'bolt://localhost:7687')
NEO4J_USER = os.getenv('NEO4J_USER', 'neo4j')
NEO4J_PASSWORD = os.getenv('NEO4J_PASSWORD', 'password')

QDRANT_URL = os.getenv('QDRANT_URL', 'http://localhost:6333')
QDRANT_API_KEY = os.getenv('QDRANT_API_KEY')
QDRANT_CHUNK_COLLECTION = os.getenv('QDRANT_CHUNK_COLLECTION', 'legal_chunks_vectors')
QDRANT_ENTITY_COLLECTION = os.getenv('QDRANT_ENTITY_COLLECTION', 'legal_entities_vectors')
QDRANT_ID_FIELD = os.getenv('QDRANT_ID_FIELD', 'id')

driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))
qdrant = QdrantClient(url=QDRANT_URL, api_key=QDRANT_API_KEY)

def run_query(query: str, params: dict | None = None):
    with driver.session() as session:
        result = session.run(query, params or {})
        return [r.data() for r in result]

schema_probe = run_query("""
MATCH (c:Chunk)
OPTIONAL MATCH (c)-[:MENTIONS]->(e:Entity)
RETURN count(DISTINCT c) AS chunks, count(DISTINCT e) AS entities, count(*) AS mention_rows
""")
schema_probe


[{'chunks': 603, 'entities': 2187, 'mention_rows': 3507}]

In [28]:
# Вытягиваем узлы/ребра и embedding_id для загрузки векторов из Qdrant
chunks = run_query("""
MATCH (c:Chunk)
RETURN
  c.chunk_id AS chunk_id,
  c.doc_id AS doc_id,
  c.embedding_id AS embedding_id,
  coalesce(c.text, '') AS text
""")

entities = run_query("""
MATCH (e:Entity)
RETURN
  e.entity_id AS entity_id,
  coalesce(e.embedding_id, e.entity_id) AS embedding_id,
  coalesce(e.entity_name, e.normalized_value, e.value, '') AS entity_text
""")

mentions = run_query("""
MATCH (c:Chunk)-[:MENTIONS]->(e:Entity)
RETURN c.chunk_id AS chunk_id, e.entity_id AS entity_id
""")

related = run_query("""
MATCH (e1:Entity)-[:RELATED_TO]->(e2:Entity)
RETURN e1.entity_id AS src_entity_id, e2.entity_id AS dst_entity_id
""")

chunks_df = pd.DataFrame(chunks).dropna(subset=['chunk_id', 'embedding_id']).drop_duplicates('chunk_id')
entities_df = pd.DataFrame(entities).dropna(subset=['entity_id', 'embedding_id']).drop_duplicates('entity_id')
mentions_df = pd.DataFrame(mentions).dropna().drop_duplicates()
related_df = pd.DataFrame(related).dropna().drop_duplicates()

print('chunks (with embedding_id):', len(chunks_df))
print('entities (with embedding_id):', len(entities_df))
print('mentions:', len(mentions_df))
print('related_to:', len(related_df))
chunks_df.head(2), entities_df.head(2), mentions_df.head(2)


Received notification from DBMS server: <GqlStatusObject gql_status='01N52', status_description='warn: property key does not exist. The property `text` does not exist. Verify that the spelling is correct.', position=<SummaryInputPosition line=7, column=14, offset=119>, raw_classification='UNRECOGNIZED', classification=<NotificationClassification.UNRECOGNIZED: 'UNRECOGNIZED'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'UNRECOGNIZED', '_severity': 'WARNING', '_position': {'offset': 119, 'line': 7, 'column': 14}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "\nMATCH (c:Chunk)\nRETURN\n  c.chunk_id AS chunk_id,\n  c.doc_id AS doc_id,\n  c.embedding_id AS embedding_id,\n  coalesce(c.text, '') AS text\n"
Received notification from DBMS server: <GqlStatusObject gql_status='01N52', status_description='warn: property key does not exist. The property `normalized_value` does not exist. Verify tha

chunks (with embedding_id): 603
entities (with embedding_id): 2187
mentions: 3507
related_to: 2293


(                               chunk_id doc_id  \
 0  019d8288-d132-71ed-b343-9c65524dcb53     12   
 1  019d8288-d136-7bfd-accb-06a906af2fe0     12   
 
                            embedding_id text  
 0  b22e00fa-af96-5b15-8d49-4f770d79cc21       
 1  d02ca84d-1bca-5a0f-8ec0-08810d645b90       ,
                                            entity_id  \
 0  f7381fc918f9933102106f5799248780039f5987967067...   
 1  8f4e8b62535fb495bb91ff066c856652416cdd676a8615...   
 
                            embedding_id        entity_text  
 0  68363780-c85c-5712-be9c-d811cab98174      Paris Commune  
 1  124ebd02-c765-5073-82f5-174cc956e4ed  Russian Civil War  ,
                                chunk_id  \
 0  019d8288-d132-71ed-b343-9c65524dcb53   
 1  019d8288-d132-71ed-b343-9c65524dcb53   
 
                                            entity_id  
 0  2246cc1e9afc11f58ec447ae8eb7cd7f6091f5b4672428...  
 1  15b38175c1f72394880594a9d93b640e5d14c9db0029ae...  )

In [29]:
# Загружаем векторы из Qdrant по embedding_id (= point.id в Qdrant)
def _extract_vector(raw_vector):
    if raw_vector is None:
        return None
    if isinstance(raw_vector, dict):
        # named vectors: берём первый
        raw_vector = next(iter(raw_vector.values())) if raw_vector else None
    if raw_vector is None:
        return None
    return np.asarray(raw_vector, dtype=np.float32)


def _normalize_qdrant_point_id(raw_id):
    if raw_id is None or pd.isna(raw_id):
        return None
    if isinstance(raw_id, (int, np.integer)):
        return int(raw_id)

    raw_id = str(raw_id).strip()
    if not raw_id:
        return None
    if raw_id.lstrip('-').isdigit():
        return int(raw_id)
    return raw_id


def fetch_vectors_by_embedding_ids(collection_name: str, embedding_ids: list[str], batch_size: int = 256):
    vectors: dict[str, np.ndarray] = {}
    ids = []
    for raw_id in embedding_ids:
        normalized_id = _normalize_qdrant_point_id(raw_id)
        if normalized_id is not None:
            ids.append(normalized_id)

    ids = list(dict.fromkeys(ids))
    for i in range(0, len(ids), batch_size):
        batch = ids[i:i+batch_size]
        points = qdrant.retrieve(
            collection_name=collection_name,
            ids=batch,
            with_payload=False,
            with_vectors=True,
        )
        for p in points:
            v = _extract_vector(p.vector)
            if v is not None:
                vectors[str(p.id)] = v
    return vectors


chunk_vec_map = fetch_vectors_by_embedding_ids(
    collection_name=QDRANT_CHUNK_COLLECTION,
    embedding_ids=chunks_df['embedding_id'].tolist(),
)
entity_vec_map = fetch_vectors_by_embedding_ids(
    collection_name=QDRANT_ENTITY_COLLECTION,
    embedding_ids=entities_df['embedding_id'].tolist(),
)

chunks_df['vec'] = chunks_df['embedding_id'].astype(str).map(chunk_vec_map)
entities_df['vec'] = entities_df['embedding_id'].astype(str).map(entity_vec_map)

chunks_df = chunks_df[chunks_df['vec'].notna()].copy().reset_index(drop=True)
entities_df = entities_df[entities_df['vec'].notna()].copy().reset_index(drop=True)

#print(entities_df)
print("ENTITIES COUNT: ", entities_df.count())

if chunks_df.empty or entities_df.empty:
    raise ValueError('Не удалось получить вектора из Qdrant. Проверьте collection names и embedding_id mapping.')

chunk_dims = chunks_df['vec'].apply(len).unique().tolist()
entity_dims = entities_df['vec'].apply(len).unique().tolist()
if len(chunk_dims) != 1 or len(entity_dims) != 1:
    raise ValueError(f'Непостоянная размерность векторов: chunk={chunk_dims}, entity={entity_dims}')

chunk_x = np.vstack(chunks_df['vec'].values).astype(np.float32)
entity_x = np.vstack(entities_df['vec'].values).astype(np.float32)

# L2-нормализация входных фичей
chunk_x = chunk_x / (np.linalg.norm(chunk_x, axis=1, keepdims=True) + 1e-12)
entity_x = entity_x / (np.linalg.norm(entity_x, axis=1, keepdims=True) + 1e-12)

chunk2idx = {cid: i for i, cid in enumerate(chunks_df['chunk_id'].tolist())}
entity2idx = {eid: i for i, eid in enumerate(entities_df['entity_id'].tolist())}

mentions_df = mentions_df[mentions_df['chunk_id'].isin(chunk2idx) & mentions_df['entity_id'].isin(entity2idx)].copy()
related_df = related_df[related_df['src_entity_id'].isin(entity2idx) & related_df['dst_entity_id'].isin(entity2idx)].copy()

mention_src = torch.tensor([chunk2idx[x] for x in mentions_df['chunk_id']], dtype=torch.long)
mention_dst = torch.tensor([entity2idx[x] for x in mentions_df['entity_id']], dtype=torch.long)

rel_src = torch.tensor([entity2idx[x] for x in related_df['src_entity_id']], dtype=torch.long) if len(related_df) else torch.empty(0, dtype=torch.long)
rel_dst = torch.tensor([entity2idx[x] for x in related_df['dst_entity_id']], dtype=torch.long) if len(related_df) else torch.empty(0, dtype=torch.long)

data = HeteroData()
data['chunk'].x = torch.from_numpy(chunk_x)
data['entity'].x = torch.from_numpy(entity_x)

data['chunk', 'mentions', 'entity'].edge_index = torch.stack([mention_src, mention_dst], dim=0)
data['entity', 'mentioned_in', 'chunk'].edge_index = torch.stack([mention_dst, mention_src], dim=0)
data['entity', 'related_to', 'entity'].edge_index = torch.stack([rel_src, rel_dst], dim=0)
data['entity', 'related_to_rev', 'entity'].edge_index = torch.stack([rel_dst, rel_src], dim=0)

print(f'Loaded Qdrant vectors: chunks={len(chunks_df)} (dim={chunk_x.shape[1]}), entities={len(entities_df)} (dim={entity_x.shape[1]})')
data


ENTITIES COUNT:  entity_id       2187
embedding_id    2187
entity_text     2187
vec             2187
dtype: int64
Loaded Qdrant vectors: chunks=603 (dim=256), entities=2187 (dim=256)


HeteroData(
  chunk={ x=[603, 256] },
  entity={ x=[2187, 256] },
  (chunk, mentions, entity)={ edge_index=[2, 3507] },
  (entity, mentioned_in, chunk)={ edge_index=[2, 3507] },
  (entity, related_to, entity)={ edge_index=[2, 2293] },
  (entity, related_to_rev, entity)={ edge_index=[2, 2293] }
)

In [30]:
# Train/Val/Test split по рёбрам MENTIONS
all_edges = data['chunk', 'mentions', 'entity'].edge_index.t().cpu().numpy()
perm = np.random.permutation(len(all_edges))
all_edges = all_edges[perm]

n = len(all_edges)
n_train = int(0.8 * n)
n_val = int(0.1 * n)

train_edges = torch.tensor(all_edges[:n_train], dtype=torch.long)
val_edges = torch.tensor(all_edges[n_train:n_train+n_val], dtype=torch.long)
test_edges = torch.tensor(all_edges[n_train+n_val:], dtype=torch.long)

# Для честной валидации/теста убираем val/test MENTIONS-рёбра из графа обучения.
train_edge_index = train_edges.t().contiguous()
train_data = data.clone()
train_data['chunk', 'mentions', 'entity'].edge_index = train_edge_index
train_data['entity', 'mentioned_in', 'chunk'].edge_index = train_edge_index.flip(0)

num_chunks = train_data['chunk'].x.shape[0]
num_entities = train_data['entity'].x.shape[0]

train_positives_by_chunk: dict[int, set[int]] = {}
for c_idx, e_idx in train_edges.tolist():
    train_positives_by_chunk.setdefault(int(c_idx), set()).add(int(e_idx))

all_positives_by_chunk: dict[int, set[int]] = {}
for c_idx, e_idx in all_edges.tolist():
    all_positives_by_chunk.setdefault(int(c_idx), set()).add(int(e_idx))


def build_hard_negative_pools(
    chunk_features: np.ndarray,
    entity_features: np.ndarray,
    positives_map: dict[int, set[int]],
    top_k: int = 64,
    batch_size: int = 128,
):
    chunk_tensor = torch.from_numpy(chunk_features)
    entity_tensor = torch.from_numpy(entity_features)
    pools: dict[int, list[int]] = {}

    for start in range(0, len(chunk_tensor), batch_size):
        stop = min(start + batch_size, len(chunk_tensor))
        scores = torch.matmul(chunk_tensor[start:stop], entity_tensor.t())
        candidate_count = min(entity_tensor.shape[0], top_k + 96)
        top_idx = torch.topk(scores, k=candidate_count, dim=1).indices.cpu().tolist()

        for local_idx, candidates in enumerate(top_idx):
            chunk_idx = start + local_idx
            positives = positives_map.get(chunk_idx, set())
            hard_candidates = []
            for entity_idx in candidates:
                entity_idx = int(entity_idx)
                if entity_idx in positives:
                    continue
                hard_candidates.append(entity_idx)
                if len(hard_candidates) == top_k:
                    break
            pools[chunk_idx] = hard_candidates

    return pools


hard_negative_top_k = 64
hard_negative_pools = build_hard_negative_pools(
    chunk_features=chunk_x,
    entity_features=entity_x,
    positives_map=train_positives_by_chunk,
    top_k=hard_negative_top_k,
)


def sample_negative_entities(
    batch_edges: torch.Tensor,
    positives_map: dict[int, set[int]],
    num_entities: int,
    hard_pools: dict[int, list[int]] | None = None,
    num_random_neg: int = 4,
    num_hard_neg: int = 2,
    max_tries: int = 50,
) -> torch.Tensor:
    total_neg = num_random_neg + num_hard_neg
    neg_entity_ids = torch.empty((len(batch_edges), total_neg), dtype=torch.long)

    for i, (c_idx, e_idx) in enumerate(batch_edges.tolist()):
        c_idx = int(c_idx)
        e_idx = int(e_idx)
        positives = positives_map.get(c_idx, set())
        selected = set()

        candidate_hards = [
            int(entity_idx)
            for entity_idx in (hard_pools or {}).get(c_idx, [])
            if int(entity_idx) != e_idx and int(entity_idx) not in positives
        ]
        hard_take = min(num_hard_neg, len(candidate_hards))
        if hard_take:
            for hard_entity in random.sample(candidate_hards, k=hard_take):
                selected.add(hard_entity)

        while len(selected) < total_neg:
            found = False
            for _ in range(max_tries):
                candidate = random.randrange(num_entities)
                if candidate != e_idx and candidate not in positives and candidate not in selected:
                    selected.add(candidate)
                    found = True
                    break
            if not found:
                break

        if len(selected) < total_neg:
            raise RuntimeError(f'Не удалось набрать {total_neg} negatives для chunk={c_idx}')

        neg_entity_ids[i] = torch.tensor(sorted(selected), dtype=torch.long)

    return neg_entity_ids

print('Hard-negative pools prepared for chunks:', len(hard_negative_pools))
len(train_edges), len(val_edges), len(test_edges)


Hard-negative pools prepared for chunks: 603


(2805, 350, 352)

In [31]:
class RetrievalHeteroSAGE(nn.Module):
    def __init__(
        self,
        hidden_dim: int = 256,
        out_dim: int = 256,
        num_layers: int = 2,
        dropout: float = 0.10,
        residual_weight: float = 0.85,
        learnable_residual: bool = True,
    ):
        super().__init__()
        if num_layers < 1:
            raise ValueError('num_layers must be >= 1')

        layer_dims = [hidden_dim] * max(num_layers - 1, 0) + [out_dim]
        self.layers = nn.ModuleList([
            HeteroConv({
                ('chunk', 'mentions', 'entity'): SAGEConv((-1, -1), layer_dim),
                ('entity', 'mentioned_in', 'chunk'): SAGEConv((-1, -1), layer_dim),
                ('entity', 'related_to', 'entity'): SAGEConv((-1, -1), layer_dim),
                ('entity', 'related_to_rev', 'entity'): SAGEConv((-1, -1), layer_dim),
            }, aggr='mean')
            for layer_dim in layer_dims
        ])
        self.dropout = dropout
        residual_logit = float(np.log(residual_weight / (1.0 - residual_weight)))
        if learnable_residual:
            self.chunk_residual_logit = nn.Parameter(torch.tensor(residual_logit, dtype=torch.float32))
            self.entity_residual_logit = nn.Parameter(torch.tensor(residual_logit, dtype=torch.float32))
        else:
            self.register_buffer('chunk_residual_logit', torch.tensor(residual_logit, dtype=torch.float32))
            self.register_buffer('entity_residual_logit', torch.tensor(residual_logit, dtype=torch.float32))

    def encode_graph(self, x_dict, edge_index_dict):
        h = {k: F.normalize(v, p=2, dim=-1) for k, v in x_dict.items()}
        for layer_idx, conv in enumerate(self.layers):
            h = conv(h, edge_index_dict)
            if layer_idx != len(self.layers) - 1:
                h = {
                    k: F.dropout(F.relu(v), p=self.dropout, training=self.training)
                    for k, v in h.items()
                }
        return {k: F.normalize(v, p=2, dim=-1) for k, v in h.items()}

    def residual_weights(self):
        return {
            'chunk': float(torch.sigmoid(self.chunk_residual_logit).detach().cpu()),
            'entity': float(torch.sigmoid(self.entity_residual_logit).detach().cpu()),
        }

    def forward(self, x_dict, edge_index_dict):
        raw = {k: F.normalize(v, p=2, dim=-1) for k, v in x_dict.items()}
        graph = self.encode_graph(raw, edge_index_dict)

        chunk_alpha = torch.sigmoid(self.chunk_residual_logit)
        entity_alpha = torch.sigmoid(self.entity_residual_logit)
        fused = {
            'chunk': F.normalize(chunk_alpha * raw['chunk'] + (1.0 - chunk_alpha) * graph['chunk'], p=2, dim=-1),
            'entity': F.normalize(entity_alpha * raw['entity'] + (1.0 - entity_alpha) * graph['entity'], p=2, dim=-1),
        }
        return fused


def edge_scores(chunk_emb: torch.Tensor, entity_emb: torch.Tensor, edges: torch.Tensor) -> torch.Tensor:
    c = chunk_emb[edges[:, 0]]
    e = entity_emb[edges[:, 1]]
    return (c * e).sum(dim=-1)


@torch.no_grad()
def collect_edge_ranks(chunk_emb: torch.Tensor, entity_emb: torch.Tensor, eval_edges: torch.Tensor) -> torch.Tensor:
    ranks = []
    for c_idx, e_idx in eval_edges.tolist():
        scores = torch.matmul(entity_emb, chunk_emb[c_idx])
        rank_pos = (torch.argsort(scores, descending=True) == e_idx).nonzero(as_tuple=False)
        if len(rank_pos):
            ranks.append(int(rank_pos[0].item()) + 1)
    return torch.tensor(ranks, dtype=torch.float32)


@torch.no_grad()
def evaluate_embeddings(chunk_emb: torch.Tensor, entity_emb: torch.Tensor, eval_edges: torch.Tensor, k_list=(5, 10, 20)):
    ranks = collect_edge_ranks(chunk_emb, entity_emb, eval_edges)
    metrics = {}
    for k in k_list:
        metrics[f'Recall@{k}'] = float((ranks <= k).float().mean().item()) if len(ranks) else 0.0
    metrics['MRR'] = float((1.0 / ranks).mean().item()) if len(ranks) else 0.0
    return metrics


@torch.no_grad()
def evaluate_retrieval(model: nn.Module, graph_data: HeteroData, eval_edges: torch.Tensor, k_list=(5, 10, 20)):
    model.eval()
    z = model(graph_data.x_dict, graph_data.edge_index_dict)
    return evaluate_embeddings(z['chunk'], z['entity'], eval_edges, k_list=k_list)


def build_metric_table(split_name: str, experiment_name: str, metrics: dict[str, float], baseline_metrics: dict[str, float]):
    rows = []
    for metric_name, metric_value in metrics.items():
        baseline_value = baseline_metrics[metric_name]
        delta_abs = metric_value - baseline_value
        delta_pct = (delta_abs / baseline_value * 100.0) if baseline_value else np.nan
        rows.append({
            'split': split_name,
            'experiment': experiment_name,
            'metric': metric_name,
            'baseline': baseline_value,
            'value': metric_value,
            'delta_abs': delta_abs,
            'delta_pct': delta_pct,
        })
    return rows


@torch.no_grad()
def embedding_diagnostics(
    chunk_emb: torch.Tensor,
    entity_emb: torch.Tensor,
    eval_edges: torch.Tensor,
    positives_map: dict[int, set[int]],
    sample_size: int = 192,
    negatives_per_edge: int = 6,
):
    eval_edges_cpu = eval_edges.detach().cpu()
    if len(eval_edges_cpu) == 0:
        return {}

    if len(eval_edges_cpu) > sample_size:
        sample_idx = torch.randperm(len(eval_edges_cpu))[:sample_size]
        sampled_edges_cpu = eval_edges_cpu[sample_idx]
    else:
        sampled_edges_cpu = eval_edges_cpu

    sampled_edges = sampled_edges_cpu.to(chunk_emb.device)
    pos_scores = edge_scores(chunk_emb, entity_emb, sampled_edges).detach().cpu()
    neg_ids = sample_negative_entities(
        sampled_edges_cpu,
        positives_map=positives_map,
        num_entities=entity_emb.shape[0],
        hard_pools=None,
        num_random_neg=negatives_per_edge,
        num_hard_neg=0,
    ).to(entity_emb.device)
    neg_scores = (entity_emb[neg_ids] * chunk_emb[sampled_edges[:, 0]].unsqueeze(1)).sum(dim=-1).detach().cpu().reshape(-1)
    ranks = collect_edge_ranks(chunk_emb, entity_emb, sampled_edges).detach().cpu()

    return {
        'pos_mean': float(pos_scores.mean().item()),
        'pos_std': float(pos_scores.std(unbiased=False).item()),
        'neg_mean': float(neg_scores.mean().item()),
        'neg_std': float(neg_scores.std(unbiased=False).item()),
        'score_gap': float(pos_scores.mean().item() - neg_scores.mean().item()),
        'median_rank': float(ranks.median().item()) if len(ranks) else np.nan,
        'p90_rank': float(torch.quantile(ranks, 0.90).item()) if len(ranks) else np.nan,
    }


In [32]:
baseline_chunk_emb = torch.from_numpy(chunk_x).to(DEVICE)
baseline_entity_emb = torch.from_numpy(entity_x).to(DEVICE)
baseline_val_metrics = evaluate_embeddings(baseline_chunk_emb, baseline_entity_emb, val_edges)
baseline_test_metrics = evaluate_embeddings(baseline_chunk_emb, baseline_entity_emb, test_edges)

train_data = train_data.to(DEVICE)
train_edges = train_edges.to(DEVICE)
val_edges = val_edges.to(DEVICE)
test_edges = test_edges.to(DEVICE)


def reset_seeds(seed: int = SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


raw_chunk_train = F.normalize(train_data['chunk'].x, p=2, dim=-1)
raw_entity_train = F.normalize(train_data['entity'].x, p=2, dim=-1)


def train_experiment(config: dict):
    reset_seeds()
    model = RetrievalHeteroSAGE(
        hidden_dim=config.get('hidden_dim', 256),
        out_dim=config.get('out_dim', 256),
        num_layers=config.get('num_layers', 2),
        dropout=config.get('dropout', 0.10),
        residual_weight=config.get('residual_weight', 0.85),
        learnable_residual=config.get('learnable_residual', True),
    ).to(DEVICE)
    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=config.get('lr', 3e-4),
        weight_decay=config.get('weight_decay', 1e-5),
    )

    max_epochs = config.get('max_epochs', 90)
    patience = config.get('patience', 12)
    best_val_mrr = float('-inf')
    best_state = None
    patience_left = patience

    for epoch in range(1, max_epochs + 1):
        model.train()
        optimizer.zero_grad()

        z = model(train_data.x_dict, train_data.edge_index_dict)
        chunk_batch = z['chunk'][train_edges[:, 0]]
        pos_entity_batch = z['entity'][train_edges[:, 1]]
        pos_scores = (chunk_batch * pos_entity_batch).sum(dim=-1, keepdim=True)

        neg_entity_ids = sample_negative_entities(
            train_edges.detach().cpu(),
            positives_map=train_positives_by_chunk,
            num_entities=num_entities,
            hard_pools=hard_negative_pools,
            num_random_neg=config.get('num_random_neg', 4),
            num_hard_neg=config.get('num_hard_neg', 2),
        ).to(DEVICE)
        neg_entity_batch = z['entity'][neg_entity_ids]
        neg_scores = (neg_entity_batch * chunk_batch.unsqueeze(1)).sum(dim=-1)

        if config.get('loss_name', 'cross_entropy') == 'pairwise_softplus':
            ranking_loss = F.softplus(neg_scores - pos_scores + config.get('margin', 0.0)).mean()
        else:
            logits = torch.cat([pos_scores, neg_scores], dim=1)
            labels = torch.zeros(logits.shape[0], dtype=torch.long, device=DEVICE)
            ranking_loss = F.cross_entropy(logits, labels)

        preserve_loss = 0.5 * (
            (chunk_batch - raw_chunk_train[train_edges[:, 0]]).pow(2).sum(dim=-1).mean()
            + (pos_entity_batch - raw_entity_train[train_edges[:, 1]]).pow(2).sum(dim=-1).mean()
        )
        loss = ranking_loss + config.get('preserve_weight', 0.0) * preserve_loss

        loss.backward()
        optimizer.step()

        val_metrics = evaluate_retrieval(model, train_data, val_edges)
        val_mrr = val_metrics['MRR']
        if val_mrr > best_val_mrr:
            best_val_mrr = val_mrr
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            patience_left = patience
        else:
            patience_left -= 1

        if epoch % 10 == 0 or epoch == 1:
            print(
                f"[{config['name']}] epoch={epoch:03d} | loss={loss.item():.4f} | "
                f"ranking_loss={ranking_loss.item():.4f} | preserve_loss={preserve_loss.item():.4f} | "
                f"val={val_metrics} | patience_left={patience_left}"
            )

        if patience_left == 0:
            print(f"[{config['name']}] early stopping at epoch {epoch:03d}; best_val_mrr={best_val_mrr:.6f}")
            break

    if best_state is not None:
        model.load_state_dict(best_state)

    model.eval()
    with torch.no_grad():
        z = model(train_data.x_dict, train_data.edge_index_dict)
        final_val_metrics = evaluate_embeddings(z['chunk'], z['entity'], val_edges)
        final_test_metrics = evaluate_embeddings(z['chunk'], z['entity'], test_edges)
        val_diag = embedding_diagnostics(z['chunk'], z['entity'], val_edges, positives_map=all_positives_by_chunk)
        test_diag = embedding_diagnostics(z['chunk'], z['entity'], test_edges, positives_map=all_positives_by_chunk)

    return {
        'name': config['name'],
        'config': config,
        'model': model,
        'val_metrics': final_val_metrics,
        'test_metrics': final_test_metrics,
        'val_diag': val_diag,
        'test_diag': test_diag,
        'residual_weights': model.residual_weights(),
    }


experiment_configs = [
    {
        'name': 'residual_two_layer_ce',
        'num_layers': 2,
        'out_dim': 256,
        'num_random_neg': 4,
        'num_hard_neg': 2,
        'loss_name': 'cross_entropy',
        'preserve_weight': 0.02,
        'residual_weight': 0.88,
        'dropout': 0.10,
    },
    {
        'name': 'residual_one_layer_ce',
        'num_layers': 1,
        'out_dim': 256,
        'num_random_neg': 4,
        'num_hard_neg': 2,
        'loss_name': 'cross_entropy',
        'preserve_weight': 0.02,
        'residual_weight': 0.88,
        'dropout': 0.05,
    },
    {
        'name': 'residual_one_layer_rankloss',
        'num_layers': 1,
        'out_dim': 256,
        'num_random_neg': 6,
        'num_hard_neg': 6,
        'loss_name': 'pairwise_softplus',
        'margin': 0.10,
        'preserve_weight': 0.03,
        'residual_weight': 0.90,
        'dropout': 0.05,
    },
]

experiment_results = []
for config in experiment_configs:
    print(f"\n=== Running {config['name']} ===")
    experiment_results.append(train_experiment(config))

best_run = max(experiment_results, key=lambda item: item['val_metrics']['MRR'])
selected_experiment = best_run
model = best_run['model']
final_val_metrics = best_run['val_metrics']
test_metrics = best_run['test_metrics']
selected_model_version = f"graphsage_{best_run['name']}"

print('\nSelected experiment:', best_run['name'])
print('Best val metrics:', final_val_metrics)
print('Test metrics:', test_metrics)
print('Residual weights:', best_run['residual_weights'])



=== Running residual_two_layer_ce ===
[residual_two_layer_ce] epoch=001 | loss=1.8672 | ranking_loss=1.8668 | preserve_loss=0.0183 | val={'Recall@5': 0.14000000059604645, 'Recall@10': 0.2028571367263794, 'Recall@20': 0.30571427941322327, 'MRR': 0.11647070944309235} | patience_left=12
[residual_two_layer_ce] epoch=010 | loss=1.8494 | ranking_loss=1.8490 | preserve_loss=0.0171 | val={'Recall@5': 0.1485714316368103, 'Recall@10': 0.22857142984867096, 'Recall@20': 0.33142855763435364, 'MRR': 0.11674996465444565} | patience_left=9
[residual_two_layer_ce] early stopping at epoch 019; best_val_mrr=0.118747

=== Running residual_one_layer_ce ===
[residual_one_layer_ce] epoch=001 | loss=1.8673 | ranking_loss=1.8669 | preserve_loss=0.0184 | val={'Recall@5': 0.1599999964237213, 'Recall@10': 0.1971428543329239, 'Recall@20': 0.31142857670783997, 'MRR': 0.11384160816669464} | patience_left=12
[residual_one_layer_ce] epoch=010 | loss=1.8511 | ranking_loss=1.8508 | preserve_loss=0.0176 | val={'Recall@

In [33]:
# Baseline без GNN и сравнение всех абляций
baseline_val_diag = embedding_diagnostics(
    baseline_chunk_emb,
    baseline_entity_emb,
    val_edges,
    positives_map=all_positives_by_chunk,
)
baseline_test_diag = embedding_diagnostics(
    baseline_chunk_emb,
    baseline_entity_emb,
    test_edges,
    positives_map=all_positives_by_chunk,
)

gnn_val_metrics = final_val_metrics
gnn_test_metrics = test_metrics

summary_rows = []
comparison_rows = []
diagnostic_rows = [
    {'experiment': 'baseline', 'split': 'val', **baseline_val_diag},
    {'experiment': 'baseline', 'split': 'test', **baseline_test_diag},
]
for result in experiment_results:
    summary_rows.append({
        'experiment': result['name'],
        'val_MRR': result['val_metrics']['MRR'],
        'test_MRR': result['test_metrics']['MRR'],
        'val_Recall@10': result['val_metrics']['Recall@10'],
        'test_Recall@10': result['test_metrics']['Recall@10'],
        'chunk_alpha': result['residual_weights']['chunk'],
        'entity_alpha': result['residual_weights']['entity'],
    })
    comparison_rows.extend(build_metric_table('val', result['name'], result['val_metrics'], baseline_val_metrics))
    comparison_rows.extend(build_metric_table('test', result['name'], result['test_metrics'], baseline_test_metrics))
    diagnostic_rows.append({'experiment': result['name'], 'split': 'val', **result['val_diag']})
    diagnostic_rows.append({'experiment': result['name'], 'split': 'test', **result['test_diag']})

summary_df = pd.DataFrame(summary_rows).sort_values(['val_MRR', 'test_MRR'], ascending=False).reset_index(drop=True)
comparison_df = pd.DataFrame(comparison_rows)
diagnostics_df = pd.DataFrame(diagnostic_rows)

print('Baseline val metrics:', baseline_val_metrics)
print('Selected GNN val metrics:', gnn_val_metrics)
print('Baseline test metrics:', baseline_test_metrics)
print('Selected GNN test metrics:', gnn_test_metrics)
print('\nMetric deltas vs baseline (% and absolute):')
print(comparison_df.round(4).to_string(index=False))
print('\nEmbedding diagnostics:')
print(diagnostics_df.round(4).to_string(index=False))
print('\nSelected experiment:', selected_experiment['name'])
summary_df


Baseline val metrics: {'Recall@5': 0.14571428298950195, 'Recall@10': 0.20000000298023224, 'Recall@20': 0.3142857253551483, 'MRR': 0.11384778469800949}
Selected GNN val metrics: {'Recall@5': 0.1599999964237213, 'Recall@10': 0.22285714745521545, 'Recall@20': 0.334285706281662, 'MRR': 0.12690195441246033}
Baseline test metrics: {'Recall@5': 0.13352273404598236, 'Recall@10': 0.2130681872367859, 'Recall@20': 0.30397728085517883, 'MRR': 0.08963935822248459}
Selected GNN test metrics: {'Recall@5': 0.1619318127632141, 'Recall@10': 0.23011364042758942, 'Recall@20': 0.3181818127632141, 'MRR': 0.10722131282091141}

Metric deltas vs baseline (% and absolute):
split                  experiment    metric  baseline  value  delta_abs  delta_pct
  val       residual_two_layer_ce  Recall@5    0.1457 0.1514     0.0057     3.9216
  val       residual_two_layer_ce Recall@10    0.2000 0.2200     0.0200    10.0000
  val       residual_two_layer_ce Recall@20    0.3143 0.3314     0.0171     5.4545
  val       

,experiment,val_MRR,test_MRR,val_Recall@10,test_Recall@10,chunk_alpha,entity_alpha
0,residual_one_layer_ce,0.126902,0.107221,0.222857,0.230114,0.879818,0.879708
1,residual_one_layer_rankloss,0.126310,0.109351,0.222857,0.232955,0.899750,0.899663
2,residual_two_layer_ce,0.118747,0.101984,0.220000,0.227273,0.880082,0.879974


In [ ]:
# Инференс эмбеддингов и запись обратно в Neo4j
# После честной оценки на train-графе считаем эмбеддинги лучшей абляции на полном графе.
data = data.to(DEVICE)
model = selected_experiment['model'].to(DEVICE)
model.eval()
with torch.no_grad():
    z = model(data.x_dict, data.edge_index_dict)

chunk_emb = z['chunk'].detach().cpu().numpy()
entity_emb = z['entity'].detach().cpu().numpy()

chunk_records = [
    {'chunk_id': cid, 'gnn_embedding': emb.tolist()}
    for cid, emb in zip(chunks_df['chunk_id'].tolist(), chunk_emb)
]
entity_records = [
    {'entity_id': eid, 'gnn_embedding': emb.tolist()}
    for eid, emb in zip(entities_df['entity_id'].tolist(), entity_emb)
]

with driver.session() as session:
    session.run("""
    UNWIND $rows AS row
    MATCH (c:Chunk {chunk_id: row.chunk_id})
    SET c.gnn_embedding = row.gnn_embedding,
        c.gnn_model_version = $model_version,
        c.gnn_updated_at = datetime()
    """, rows=chunk_records, model_version=selected_model_version)

    session.run("""
    UNWIND $rows AS row
    MATCH (e:Entity {entity_id: row.entity_id})
    SET e.gnn_embedding = row.gnn_embedding,
        e.gnn_model_version = $model_version,
        e.gnn_updated_at = datetime()
    """, rows=entity_records, model_version=selected_model_version)

print('Selected experiment for upsert:', selected_experiment['name'])
print('Embeddings upserted:', len(chunk_records), 'chunks,', len(entity_records), 'entities')


In [ ]:
# Demo retrieval: top-N entities для произвольного chunk
sample_chunk_idx = 0
sample_chunk_id = chunks_df.iloc[sample_chunk_idx]['chunk_id']
sample_text = chunks_df.iloc[sample_chunk_idx]['text'][:220]

data = data.to(DEVICE)
model = selected_experiment['model'].to(DEVICE)
with torch.no_grad():
    z = model(data.x_dict, data.edge_index_dict)
    scores = torch.matmul(z['entity'], z['chunk'][sample_chunk_idx])
    topk = torch.topk(scores, k=min(10, len(scores))).indices.cpu().tolist()

print('Experiment:', selected_experiment['name'])
print('Chunk:', sample_chunk_id)
print('Text preview:', sample_text)
print('Top entities:')
for i, idx in enumerate(topk, 1):
    row = entities_df.iloc[idx]
    print(f'{i:02d}. {row.entity_id} | {row.entity_text}')


In [ ]:
driver.close()
